eid data source: https://www.pxweb.bfs.admin.ch/pxweb/fr/px-x-1703030000_100/-/px-x-1703030000_100.px/table/tableViewLayout2/ $\newline$
2025 population data source: https://www.pxweb.bfs.admin.ch/pxweb/fr/px-x-0102020000_202/-/px-x-0102020000_202.px/
$\newline$
Note: this data analysis was done without any help from AI.


In [368]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import statsmodels.formula.api as smf

In [369]:
cantons_df = pd.read_csv("cantons_data.csv")
eid_df = pd.read_csv("eid_votation.csv")

In [370]:
cantons_df = cantons_df[cantons_df["Sex"] != "Sexe - total"]
cantons_df = cantons_df[cantons_df["Age"] != "100 ans ou plus"]
cantons_df = cantons_df[cantons_df["Canton"] != "Suisse"]
cantons_df = cantons_df[cantons_df["Canton"] != "Sans indication"]
col = ["Canton", "Age", "Sex", "Count", "Immigration", "Swiss citizenship acquisition"]
cantons_df = cantons_df[col]

cantons_df.head()

,Canton,Age,Sex,Count,Immigration,Swiss citizenship acquisition
332,Z�rich,18 ans,Homme,6227,29,106
333,Z�rich,19 ans,Homme,5993,34,74
334,Z�rich,20 ans,Homme,6065,39,63
335,Z�rich,21 ans,Homme,5925,32,42
336,Z�rich,22 ans,Homme,5819,38,28


In [371]:
cantons_df["Canton"].unique()

array(['Z�rich', 'Bern / Berne', 'Luzern', 'Uri', 'Schwyz', 'Obwalden',
       'Nidwalden', 'Glarus', 'Zug', 'Fribourg / Freiburg', 'Solothurn',
       'Basel-Stadt', 'Basel-Landschaft', 'Schaffhausen',
       'Appenzell Ausserrhoden', 'Appenzell Innerrhoden', 'St. Gallen',
       'Graub�nden / Grigioni / Grischun', 'Aargau', 'Thurgau', 'Ticino',
       'Vaud', 'Valais / Wallis', 'Neuch�tel', 'Gen�ve', 'Jura'],
      dtype=object)

In [372]:
def fix_cantons(cantons_df):
    cantons_df = cantons_df.replace('Z�rich','Zurich')
    cantons_df = cantons_df.replace('Bern / Berne','Berne')
    cantons_df = cantons_df.replace('Fribourg / Freiburg','Fribourg')
    cantons_df = cantons_df.replace('Graub�nden / Grigioni / Grischun','Grischun')
    cantons_df = cantons_df.replace('Valais / Wallis','Valais')
    cantons_df = cantons_df.replace('Neuch�tel','Neuchatel')
    cantons_df = cantons_df.replace('Gen�ve','Geneve')

    return cantons_df

cantons_df = fix_cantons(cantons_df)
eid_df = fix_cantons(eid_df)



In [373]:
cantons_df["Canton"].unique()

array(['Zurich', 'Berne', 'Luzern', 'Uri', 'Schwyz', 'Obwalden',
       'Nidwalden', 'Glarus', 'Zug', 'Fribourg', 'Solothurn',
       'Basel-Stadt', 'Basel-Landschaft', 'Schaffhausen',
       'Appenzell Ausserrhoden', 'Appenzell Innerrhoden', 'St. Gallen',
       'Grischun', 'Aargau', 'Thurgau', 'Ticino', 'Vaud', 'Valais',
       'Neuchatel', 'Geneve', 'Jura'], dtype=object)

In [374]:
eid_df.head()

,Canton,Voter registrations,Vote count,Participation %,Valid votes,Yes,No,yes_percentage
0,Suisse,5641040,2796897,49.58,2747948,1384586,1363362,50.39
1,Zurich,976490,509297,52.16,503923,274702,229221,54.51
2,Berne,750745,358855,47.80,354158,171771,182387,48.50
3,Luzern,288363,153604,53.27,151603,80261,71342,52.94
4,Uri,27226,12731,46.76,12567,5109,7458,40.65


In [375]:
cantons_df.head()

,Canton,Age,Sex,Count,Immigration,Swiss citizenship acquisition
332,Zurich,18 ans,Homme,6227,29,106
333,Zurich,19 ans,Homme,5993,34,74
334,Zurich,20 ans,Homme,6065,39,63
335,Zurich,21 ans,Homme,5925,32,42
336,Zurich,22 ans,Homme,5819,38,28


In [376]:
# for later graphs that have to do with age groups

age_groups = [range(18, 24), range(24,30), range(30, 36), range(36, 42), range(42, 48), range(48, 54), range(54, 60), range(60, 66),
              range(66, 72), range(72, 78), range(78, 84), range(84, 90), range(90, 96)]
def age_sort(df, age_groups):
    for i in range(95, 100):
        df = df[df["Age"] != str(i) + " ans"]
    # group by ages to count population groups
    for age_group in age_groups:
        new_string = str(age_group[0]) + " - " + str(age_group[-1])
        for age in age_group:
            age_string = str(age) + " ans"
            df = df.replace(age_string, new_string)
    df.reset_index()
    return df
        

In [377]:
def age_transform(df):
    for i in range(18, 100):
        age_string = str(i) + " ans"
        df = df.replace(age_string, i)
    return df

In [378]:
# represent demographics with percentages instead of count
def canton_proportions(df):
    df["Count"] = df["Count"] / df.groupby("Canton")["Count"].transform('sum')
    return df

In [379]:
cantons_df = canton_proportions(cantons_df)
cantons_df = cantons_df.rename(columns={"Count":"Proportion"})
cantons_df = cantons_df.replace("Homme","Male")
cantons_df = cantons_df.replace("Femme","Female")
cantons_df.head()

,Canton,Age,Sex,Proportion,Immigration,Swiss citizenship acquisition
332,Zurich,18 ans,Male,0.006559,29,106
333,Zurich,19 ans,Male,0.006313,34,74
334,Zurich,20 ans,Male,0.006389,39,63
335,Zurich,21 ans,Male,0.006241,32,42
336,Zurich,22 ans,Male,0.006129,38,28


In [380]:
#cantons_df = age_sort(cantons_df, age_groups)
cantons_df = age_transform(cantons_df)
cantons_df = cantons_df.groupby(["Canton", "Age", "Sex"]).sum()
cantons_df.head(10)

/var/folders/22/_dymvqsd4tgfb0zzj_kpnj340000gn/T/ipykernel_2023/749895442.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace(age_string, i)


Proportion  Immigration  Swiss citizenship acquisition
Canton Age Sex                                                           
Aargau 18  Female    0.005881            4                             27
           Male      0.006156           11                             33
       19  Female    0.005886            9                             26
           Male      0.006318           11                             30
       20  Female    0.005799           10                             19
           Male      0.006334            2                             18
       21  Female    0.005981            8                             16
           Male      0.006297           12                             18
       22  Female    0.005831           10                             22
           Male      0.006274            7                             15

In [381]:
cantons_df = cantons_df.reset_index()
cantons_df.pivot(columns=["Age", "Sex"], index="Canton", values=["Proportion"])

Proportion                                          \
Age                            18                  19                  20   
Sex                        Female      Male    Female      Male    Female   
Canton                                                                      
Aargau                   0.005881  0.006156  0.005886  0.006318  0.005799   
Appenzell Ausserrhoden   0.005432  0.006064  0.005827  0.006381  0.005405   
Appenzell Innerrhoden    0.006594  0.006343  0.005175  0.006594  0.005509   
Basel-Landschaft         0.005809  0.005951  0.005666  0.005534  0.005698   
Basel-Stadt              0.005025  0.005493  0.005121  0.005454  0.005331   
Berne                    0.005613  0.005788  0.005465  0.005871  0.005518   
Fribourg                 0.006465  0.006671  0.006451  0.006859  0.006263   
Geneve                   0.007414  0.007885  0.007327  0.007719  0.007933   
Glarus                   0.004989  0.006255  0.005296  0.005948  0.005027   
Grischun                 0.005218  0.005096  0.005125  0.005182  0.004937   
Jura                     0.005981  0.005866  0.006020  0.007285  0.006633   
Luzern                   0.006092  0.006218  0.006152  0.006085  0.005775   
Neuchatel                0.006606  0.006670  0.007120  0.007422  0.007028   
Nidwalden                0.005045  0.005360  0.005392  0.005770  0.005329   
Obwalden                 0.006249  0.007016  0.005627  0.006249  0.006176   
Schaffhausen             0.005467  0.005882  0.005071  0.005486  0.005674   
Schwyz                   0.005402  0.005820  0.005662  0.005774  0.005616   
Solothurn                0.005433  0.005229  0.005025  0.005417  0.005251   
St. Gallen               0.006020  0.006214  0.005842  0.006417  0.006024   
Thurgau                  0.006320  0.006287  0.005726  0.006203  0.005738   
Ticino                   0.005890  0.006353  0.005913  0.006533  0.006089   
Uri                      0.005848  0.006107  0.005811  0.006662  0.005441   
Valais                   0.005462  0.005872  0.005716  0.006113  0.005777   
Vaud                     0.007085  0.007535  0.007096  0.007645  0.007078   
Zug                      0.005962  0.005898  0.006411  0.006411  0.005770   
Zurich                   0.006100  0.006559  0.006137  0.006313  0.006096   

                                                                          ...  \
Age                                     21                  22            ...   
Sex                         Male    Female      Male    Female      Male  ...   
Canton                                                                    ...   
Aargau                  0.006334  0.005981  0.006297  0.005831  0.006274  ...   
Appenzell Ausserrhoden  0.006275  0.005194  0.004772  0.005511  0.005273  ...   
Appenzell Innerrhoden   0.005843  0.006010  0.005425  0.005843  0.005759  ...   
Basel-Landschaft        0.006020  0.005708  0.005703  0.005291  0.005698  ...   
Basel-Stadt             0.005702  0.005521  0.005540  0.005454  0.005617  ...   
Berne                   0.005702  0.005612  0.005725  0.005496  0.005580  ...   
Fribourg                0.006732  0.006676  0.006920  0.006742  0.006723  ...   
Geneve                  0.007889  0.008126  0.007960  0.007486  0.007351  ...   
Glarus                  0.005680  0.005526  0.005066  0.004912  0.006179  ...   
Grischun                0.005276  0.005261  0.005477  0.005348  0.005880  ...   
Jura                    0.006882  0.006346  0.006748  0.006595  0.006710  ...   
Luzern                  0.006232  0.005969  0.006123  0.005694  0.005941  ...   
Neuchatel               0.007422  0.006881  0.007468  0.006918  0.007065  ...   
Nidwalden               0.005581  0.004824  0.005486  0.005013  0.005959  ...   
Obwalden                0.005189  0.006212  0.005920  0.005591  0.005701  ...   
Schaffhausen            0.005486  0.005090  0.005411  0.005542  0.005693  ...   
Schwyz                  0.006081  0.005430  0.006090  0.005467  0.005858  ...   
Solothurn  

In [382]:
eid_df = eid_df.groupby(["Canton"]).sum()
eid_df.head(10)

,Voter registrations,Vote count,Participation %,Valid votes,Yes,No,yes_percentage
Canton,,,,,,,
Aargau,447798,232292,51.87,230470,113121,117349,49.08
Appenzell Ausserrhoden,39526,21896,55.40,21652,9949,11703,45.95
Appenzell Innerrhoden,12385,6006,48.49,5859,2644,3215,45.13
Basel-Landschaft,192244,96303,50.09,93940,46245,47695,49.23
Basel-Stadt,114378,55409,48.44,54544,30976,23568,56.79
Berne,750745,358855,47.80,354158,171771,182387,48.50
Fribourg,219163,100846,46.01,98853,49792,49061,50.37
Geneve,285837,119086,41.66,112906,62308,50598,55.19
Glarus,26836,12159,45.31,12008,4965,7043,41.35


In [383]:
data_df = pd.merge(left=eid_df, right=cantons_df, left_on="Canton", right_on="Canton", how="inner")
data_df

,Canton,Voter registrations,Vote count,Participation %,Valid votes,Yes,No,yes_percentage,Age,Sex,Proportion,Immigration,Swiss citizenship acquisition
0,Aargau,447798,232292,51.87,230470,113121,117349,49.08,18,Female,0.005881,4,27
1,Aargau,447798,232292,51.87,230470,113121,117349,49.08,18,Male,0.006156,11,33
2,Aargau,447798,232292,51.87,230470,113121,117349,49.08,19,Female,0.005886,9,26
3,Aargau,447798,232292,51.87,230470,113121,117349,49.08,19,Male,0.006318,11,30
4,Aargau,447798,232292,51.87,230470,113121,117349,49.08,20,Female,0.005799,10,19
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4259,Zurich,976490,509297,52.16,503923,274702,229221,54.51,97,Male,0.000181,0,0
4260,Zurich,976490,509297,52.16,503923,274702,229221,54.51,98,Female,0.000400,0,0
4261,Zurich,976490,509297,52.16,503923,274702,229221,54.51,98,Male,0.000134,0,0
4262,Zurich,976490,509297,52.16,503923,274702,229221,54.51,99,Female,0.000253,0,0


In [386]:
model = smf.ols(formula='yes_percentage ~ Age*C(Sex) + Immigration', data=data_df).fit()

In [387]:
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:         yes_percentage   R-squared:                       0.225
Model:                            OLS   Adj. R-squared:                  0.224
Method:                 Least Squares   F-statistic:                     308.6
Date:                Mon, 27 Apr 2026   Prob (F-statistic):          2.14e-233
Time:                        17:25:25   Log-Likelihood:                -12172.
No. Observations:                4264   AIC:                         2.435e+04
Df Residuals:                    4259   BIC:                         2.439e+04
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             44.3248      0